# Results overview


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reported_runs/analysis").is_dir())
sys.path.insert(0, str(repo)) if str(repo) not in sys.path else None
from reported_runs.analysis.presentation import load_results, NotebookReport, temporal_table, effect_table, rank_summary, unique_trajectories, label_conditions
from reported_runs.analysis.plotting import trajectory_grid, effect_grid, rank_grid, association_grid
from reported_runs.analysis.comparisons import summarise_iterations
%matplotlib inline

results = load_results()
report = NotebookReport(results, 'analysis')
scales = report.scales
trajectory = label_conditions(results["trajectory"])
sources = results["sources"]
summaries = results["summaries"]
effect_summaries = results["effect_summaries"]
pd.set_option("display.max_columns", 18)
pd.set_option("display.max_rows", 60)

## Snapshot and run status


In [ ]:
status = sources.groupby(['dataset', 'study', 'status'], dropna=False).agg(logical_cells=('source_id', 'size'), physical_trajectories=('source_id', 'nunique')).reset_index()
report.table(status, 'analysis_source_status', "")
display(Markdown(f"Snapshot: `{results.metadata['snapshot_id']}`; {results.metadata['completed_physical_runs']} completed physical trajectories; {results.metadata['pending_logical_cells']} pending logical cells."))

In [ ]:
pending = sources[sources['status'].ne('complete')]
report.table(pending[['study', 'seed', 'model', 'intervention', 'status']], 'analysis_pending_sources', "")

## Metric and plotting conventions


## Reproducibility checks


In [ ]:
health = results['health'].drop_duplicates('physical_run_id')
report.table(health[['dataset', 'seed', 'physical_run_id', 'events_sha256_verified', 'diagnostics_sha256_verified', 'canonical_opportunities', 'iterations']], 'analysis_verified_sources', "")

## LLM replica agreement across all recorded cases


In [ ]:
from reported_runs.analysis.replica_agreement import build_replica_agreement
from reported_runs.analysis.replica_agreement_views import compact_replica_table, TABLE_NAME, TABLE_COLUMN_FORMAT, COUNT_EXPORT_NAME

replica_runs = build_replica_agreement(results['probes'], sources)
replica_table = compact_replica_table(replica_runs)
replica_csv = report.exports / 'data' / COUNT_EXPORT_NAME
replica_csv.parent.mkdir(parents=True, exist_ok=True)
replica_runs.to_csv(replica_csv, index=False)
report.table(replica_table, TABLE_NAME, "", source_frame=replica_runs[replica_runs.replica_status.ne('not_recorded')], column_format=TABLE_COLUMN_FORMAT)
display(Markdown(f'[run CSV]({replica_csv.as_posix()})'))

## Suggested main-text figures


## Exports and refreshing


In [ ]:
display(Markdown(f"**Exports:** `{report.exports}`\n\n**Metric version:** `{results.metadata['metric_version']}`\n\n**Draft source:** `{results.metadata['draft_path']}`"))

## Portable replay scope
